# HarborMind Data Pipeline - Phase 2: Feature Engineering (Silver to Gold Layer)

**Objective:**
Transform the raw, noisy AIS pings from the Bronze Delta Table into an ML-ready, highly enriched Gold dataset.

**Core Operations:**
1. **Zero-Data-Loss Imputation:** 3-layer Waterfall strategy for missing dimensions.
2. **Spatio-Temporal Kinematics:** Haversine distances, Heading Errors, and Acceleration calculations.
3. **Target Label Definition:** Calculating port congestion delays (`delay_minutes`) based on robust mooring logic.

**Author:** Kelvin Nguyen


In [ ]:
# PARAMETER
batch_name = "2023H2"


In [ ]:
from pyspark.sql.functions import col, when, count, isnan, lit
from pyspark.sql import Window
import pyspark.sql.functions as F

import math

In [ ]:
df = spark.read.format("delta").table("ais_zone11_bronze") \
    .filter(F.col("batch") == batch_name) \
    .drop("batch")
print(f"Batch {batch_name}: {df.count()} rows")

## 1. Data Cleaning & Waterfall Imputation
AIS sensors frequently drop static data (Length, Width, Draft). We use a 3-layer waterfall approach to impute missing values without losing rows:
1. **Transparency Flags:** Mark imputed values for the XGBoost model.
2. **Intra-vessel Forward/Backward Fill:** Borrow missing values from the ship's own historical/future pings.
3. **Inter-vessel Group Median:** Fallback to the median draft/length of similar vessel types.

### Zone port at Los Angeles & Long Beach

In [ ]:
df_zone11 = df.filter(
    (col("latitude").between(33.5, 33.9)) & 
    (col("longitude").between(-118.5, -118.0)) &
    (col("vessel_type").between(70, 89))
)

In [ ]:
# row_count = df_zone11.count()
# col_count = len(df_zone11.columns)

# print(f"Shape: ({row_count}, {col_count})")

### Check Null value

In [ ]:
# # df.limit(5).show(truncate=False)
# column_types = dict(df_zone11.dtypes)

# null_analysis = df_zone11.select([
#     count(
#         when(
#             col(c).isNull() | 
#             (isnan(col(c)) if column_types[c] in ("double", "float") else lit(False)), 
#             c
#         )
#     ).alias(c) 
#     for c in df_zone11.columns
# ])



### Fill null value

#### 1. Transparency: 
- Create the flag before filling anything. This is crucial because it tells XGBoost model which missing values we have handle

#### 2. Intra-vessel Consistency(forward/backward fill)

- A ship's draft is a semi-static feature. It physically cannot change while the ship is sailing; it only changes when cargo is loaded or unloaded at a port. Therefore, if a ship misses a ping at 9:00, 'borrowing' its own draft from 8:00 (forward fill) or 10:00 (backward fill) is extremely accurate.

In [ ]:
window_ffill = Window.partitionBy("mmsi").orderBy("base_date_time") \
              .rowsBetween(Window.unboundedPreceding, Window.currentRow)

window_bfill = Window.partitionBy("mmsi").orderBy("base_date_time") \
              .rowsBetween(Window.currentRow, Window.unboundedFollowing)
              
df_silver = df_zone11 \
    .withColumn("draft_imputed_flag", F.when(F.col("draft").isNull(), 1).otherwise(0)) \
    .withColumn("fwd_filled", F.last("draft", ignorenulls=True).over(window_ffill)) \
    .withColumn("bwd_filled", F.first("draft", ignorenulls=True).over(window_bfill)) \
    .withColumn("draft_filled", F.coalesce(F.col("fwd_filled"), F.col("bwd_filled"))) \
    .drop("fwd_filled", "bwd_filled")

#### 3. Inter-vessel Probability(grouped median)

- If a ship's transponder is broken and it never broadcasts its draft all day, cannot borrow from its past. Falling back to the Median draft of its specific vessel_type (e.g., matching it with other Cargo Ships) is the safest statistical guess that avoids being skewed by massive outlier ships.

In [ ]:
check_fill = df_silver.select(
    F.count(F.when(F.col("draft").isNull(), 1)).alias("Original_Nulls"),
    F.count(F.when(F.col("draft_filled").isNull(), 1)).alias("Remaining_Nulls")
)

check_fill.show()

In [ ]:
median_draft_df = df_silver.filter(F.col("draft").isNotNull()) \
    .groupBy("vessel_type") \
    .agg(F.percentile_approx("draft", 0.5).alias("median_draft_by_type"))
df_silver = df_silver.join(median_draft_df, on="vessel_type", how="left")
df_silver = df_silver.withColumn(
    "draft_final",
    F.coalesce(F.col("draft_filled"), F.col("median_draft_by_type"))
)

In [ ]:
final_check = df_silver.select(
    F.count(F.when(F.col("draft").isNull(), 1)).alias("Original_Nulls"),
    F.count(F.when(F.col("draft_final").isNull(), 1)).alias("Final_Remaining_Nulls")
)
final_check.show()

In [ ]:
df_silver_final = df_silver.withColumn("draft", F.col("draft_final")) \
                           .select(
                               *[c for c in df_zone11.columns if c != 'draft'], # Lấy tất cả cột cũ trừ draft cũ
                               "draft", 
                               "draft_imputed_flag"
                           )

df_silver_final.limit(5).show()

In [ ]:
for col_name in ["length", "width", "sog", "cog"]:
    df_silver = df_silver.withColumn(
        f"{col_name}_imputed_flag", 
        F.when(F.col(col_name).isNull(), 1).otherwise(0)
    )

for col_name in ["length", "width"]:
    df_silver = df_silver.withColumn(f"{col_name}_fwd", F.last(col_name, ignorenulls=True).over(window_ffill))
    df_silver = df_silver.withColumn(f"{col_name}_bwd", F.first(col_name, ignorenulls=True).over(window_bfill))
    
    df_silver = df_silver.withColumn(
        f"{col_name}_filled", 
        F.coalesce(F.col(f"{col_name}_fwd"), F.col(f"{col_name}_bwd"))
    ).drop(f"{col_name}_fwd", f"{col_name}_bwd")

median_dims = df_silver.filter(F.col("length").isNotNull()) \
    .groupBy("vessel_type") \
    .agg(
        F.percentile_approx("length", 0.5).alias("med_length"),
        F.percentile_approx("width", 0.5).alias("med_width")
    )
df_silver = df_silver.join(median_dims, on="vessel_type", how="left")
df_silver = df_silver.withColumn("length_final", F.coalesce(F.col("length_filled"), F.col("med_length")))
df_silver = df_silver.withColumn("width_final", F.coalesce(F.col("width_filled"), F.col("med_width")))

In [ ]:
for col_name in ["sog", "cog"]:
    df_silver = df_silver.withColumn(f'{col_name}_fwd', F.last(col_name, ignorenulls=True).over(window_ffill))
    df_silver = df_silver.withColumn(f'{col_name}_bwd', F.first(col_name,ignorenulls=True).over(window_bfill))
    df_silver = df_silver.withColumn(
        f"{col_name}_final", 
        F.coalesce(F.col(f"{col_name}_fwd"), F.col(f"{col_name}_bwd"), F.lit(0.0))
    ).drop(f"{col_name}_fwd", f"{col_name}_bwd")

In [ ]:
final_cols = {
    "length": "length_final", 
    "width": "width_final", 
    "sog": "sog_final", 
    "cog": "cog_final",
    "draft": "draft_final"
}
df_silver_clean = df_silver
for original, final in final_cols.items():
    if final in df_silver_clean.columns:
        df_silver_clean = df_silver_clean.withColumn(original, F.col(final))

cols_to_keep = df_zone11.columns + [c for c in df_silver_clean.columns if "flag" in c]
df_silver_clean = df_silver_clean.select(cols_to_keep)

null_counts = df_silver_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in ["length", "width", "sog", "cog", "draft"]
])
# null_counts.show()

In [ ]:
# 1. SOG validation (ITU-R M.1371 standard)
df_silver_clean = df_silver_clean.withColumn("sog",
    F.when(F.col("sog") > 30, None).otherwise(F.col("sog")))

In [ ]:
row_count = df_silver_clean.count()
col_count = len(df_silver_clean.columns)

print(f"Shape: ({row_count}, {col_count})")

## 2. Feature Engineering: Static & Temporal
- **Static:** Calculate physical footprint (`vessel_area`) and hydrodynamic profile (`dimension_ratio`).
- **Temporal:** Extract cyclical human-schedule features (hour, day, month) using Sine/Cosine transformations to preserve continuity (e.g., Hour 23 is close to Hour 0).


#### 2.1. Static Features

vessel_area, dimension_ratio, vessel_size_category

vessel_area: 2D footprint or suface area of the ship 

    - Why did we need it?
        - A port has limited physical space (Berths, anchorage zones, channels). A ship that occupies 10k m^2 takes up more that scarce resource than a ship taking up 500 m^2.
dimension_ratio: How "slender" or "boxy" the ship is

    - Why did we need it?
        - THis relates to ship design and hydrodynamics.
        - High ratio (long & narrow): often fast ship or specific tankers
        - Low ratio (boxy): Often bulk carriers or specialized cargo

vessel_size_category: Human-readable grouping of sizes baed on length (redundant)
    
    - Why did we need it?
        - Maritime traffic coltrol actually uses these classifications
        - Tree-Based models handle continuous number very well. So, grouping them into categories might seem redundant? 
        - However, creating explicit bins helps the model capture non-linear jumps. For example, the delay for ships under 200m might be relatively uniform, but the delay might spike dramatically the moment a ship crosses  300m threshold (because only 1 or 2 berths can handle it). Providing the category makes this threshold explicitly clear to the model

draft_to_length_ratio: How deep the bottom of the ship reaches underwater.

    - Why do we need it: This ratio acts as a "Is this ship full or empty?" indicator. Imagine two Cargo ships that are both 200 meters long (same vessel_area, same vessel_size_category):

        Ship A has a draft of 14 meters. (High Ratio)

        Ship B has a draft of 6 meters. (Low Ratio)

        Even though they are the exact same size, Ship A is fully loaded with cargo, pushing it deep into the water. Ship B is completely empty.


In [ ]:
df_silver_clean.columns

In [ ]:
df_silver_clean = df_silver_clean \
    .withColumn('vessel_area', 
        F.when((F.col("length") > 0) & (F.col("width") > 0), 
               F.col("length") * F.col("width"))
    ) \
    .withColumn('dimension_ratio', 
        F.when(F.col('width') > 0, 
               F.col('length') / F.col('width'))
    ) \
    .withColumn('draft_to_length_ratio', 
        F.when(F.col('length') > 0, 
               F.col('draft') / F.col('length'))
    ) 
    # \
    # .withColumn('vessel_size_category', 
    #     F.when(F.col('length').isNull() | (F.col('length') <= 0), 'Unknown')
    #      .when(F.col('length') < 200, 'Feeder')
    #      .when(F.col('length') <= 294.13, 'Panamax')
    #      .when(F.col('length') <= 366.0, 'Neo-Panamax')
    #      .otherwise('ULCV')
    # )


"vessel_size_category" -> remember to convert to categorical type

In [ ]:
# display(df_silver_clean.limit(5))

#### 2.2 Temporal Features

- [ ]  Hour of Day (sin_hour and cos_hour)
- [ ]  Day of Week
- [ ]  Month

In [ ]:
#raw extraction
df_silver_clean = df_silver_clean.withColumn('hour', F.hour(F.col('base_date_time'))) \
                .withColumn('day_of_week', F.dayofweek(F.col('base_date_time'))) \
                .withColumn('month', F.month(F.col('base_date_time')))
#cyclical convertion
df_silver_clean = df_silver_clean \
    .withColumn("hour_sin", F.sin(2 * math.pi * F.col('hour') / 24)) \
    .withColumn('hour_cos', F.cos(2 * math.pi * F.col('hour') / 24))

df_silver_clean = df_silver_clean \
    .withColumn("day_of_week_sin", F.sin(2 * math.pi * F.col('day_of_week') / 7)) \
    .withColumn('day_of_week_cos', F.cos(2 * math.pi * F.col('day_of_week') / 7))

df_silver_clean = df_silver_clean \
    .withColumn("month_sin", F.sin(2 * math.pi * F.col('month') / 12)) \
    .withColumn('month_cos', F.cos(2 * math.pi * F.col('month') / 12))


human schedules: is_weekend & is_night_shirt (binary shortcut)

In [ ]:
df_silver_clean = (
    df_silver_clean
    .withColumn("is_weekend", 
                F.when(F.col("day_of_week").isin(1, 7), 1).otherwise(0))
    .withColumn("is_night_shift", 
                F.when((F.col("hour") < 6) | (F.col("hour") >= 18), 1).otherwise(0))
    .withColumn("is_gate_hours", 
                F.when((F.col("hour").between(7, 16)) & (F.col("is_weekend") == 0), 1).otherwise(0))
)

In [ ]:
display(df_silver_clean.limit(5))

## 3. Feature Engineering: Geospatial (Space-Based)
Calculate physical relationships between the ship and the Port of LA/LB.
- **Distance to Port:** Calculated using the Haversine formula (Earth's curvature).
- **Heading Error:** The deviation between the ship's current Course Over Ground (COG) and the True Bearing to the port.


Notes:

Distance to Port: (Crucial) This is my primary predictor. A ship 50km away has a very different priority than a ship 2km away.

Is in Waiting Area: (Essential) This is likely my Target Label or a key feature for identifying the "Queue." Without this, you don't know who is actually "waiting" versus who is just passing through the zone.

Bearing to Port (heading Error): Imagine two ships are both 5km from the port. Ship A is pointing directly at the port. Ship B is pointing away from the port. Even though their distance is the same, Ship B is clearly not arriving soon. Adding a feature that calculates the difference between the ship's cog (Course Over Ground) and the bearing to the port tells the model if the ship is actually "targeting" the dock.

Distance to Port: Haversine formula

$\Delta \text{lat} = \text{lat}_2 - \text{lat}_1$

$\Delta \text{lon} = \text{lon}_2 - \text{lon}_1$

$a = \sin^2(\Delta \text{lat}/2) + \cos(\text{lat}_1) \cdot \cos(\text{lat}_2) \cdot \sin^2(\Delta \text{lon}/2)$

$c = 2 \cdot \operatorname{atan2}(\sqrt{a}, \sqrt{1-a})$

$d = \text{RADIUS} \cdot c$
Source: 

[Scikit-Learn Documentation: Distance Metrics](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.DistanceMetric.html)

[PostGIS Documentation: ST_Distance](https://postgis.net/docs/ST_Distance.html)


Notes: 
Haversine is not 100% perfect.

Haversine assumes the Earth is a perfect sphere. In reality, the Earth is an "Oblate Spheroid" (it bulges slightly at the equator like a squished basketball). Because of this, Haversine has an error rate of about 0.5%.

There is an even more advanced formula called Vincenty's formulae, which calculates distance to the exact millimeter. But Vincenty's requires complex, iterative loops that would completely crash a PySpark cluster processing millions of rows.


In [ ]:
RADIUS = 6371 # (km)
P_LAT = math.radians(33.72)
P_LON = math.radians(-118.26)

df_silver_clean = df_silver_clean \
    .withColumn('lat_rad', F.radians(F.col('latitude'))) \
    .withColumn('lon_rad', F.radians(F.col('longitude')))

df_silver_clean = df_silver_clean \
    .withColumn('dlat', F.col('lat_rad') - F.lit(P_LAT)) \
    .withColumn('dlon', F.col('lon_rad') - F.lit(P_LON))

df_silver_clean = df_silver_clean.withColumn('a', 
    F.sin(F.col('dlat') / 2)**2 + 
    F.cos(F.col('lat_rad')) * F.cos(F.lit(P_LAT)) * F.sin(F.col('dlon') / 2)**2
)
df_silver_clean = df_silver_clean.withColumn('c', 
    2 * F.atan2(F.sqrt(F.col('a')), F.sqrt(1 - F.col('a')))
)

df_silver_clean = df_silver_clean.withColumn('distance_to_port', F.col('c') * RADIUS)

In [ ]:
display(df_silver_clean.limit(5))

Is in Waiting Area (in_waiting_area): True/False - the ship is inside Geofence & speed < 0.5 knots & status == 1 → waiting

Explain for logic of this feature:

1. **Why $< 0.5$ Knots?** We cannot use Speed == 0 for ships. GPS sensors have a margin of error (GPS drift), and anchored ships are pushed by wind, tides, and currents around their anchor chain (Anchor Swing). If we wait for a ship's speed to hit exactly 0.0, our model will think every ship is constantly moving.

Citations:

Academic Standard for AIS Trajectory Mining:In the foundational paper "Vessel Pattern Knowledge Discovery from AIS Data" by Pallotta et al. (NATO Science and Technology Organization), the algorithm specifically defines a vessel as "Stationary" (at anchor or moored) when the Speed Over Ground (SOG) falls below 0.5 knots.

    Source: Pallotta, G., Vespe, M., & Bryan, K. (2013). Vessel Pattern Knowledge Discovery from AIS Data: A Framework for Anomaly Detection and Route Prediction. Sensors.

Commercial AIS Data Providers:The world's largest commercial AIS trackers (like MarineTraffic and Spire Maritime) use $< 0.5$ knots as the hard threshold in their APIs to classify a ship's state as "Stopped" or "Anchored" versus "Underway."

    Source: MarineTraffic API Documentation / Spire Maritime Data Dictionary.

2. **Why 3 km to 20 km?**

This threshold is specific to the geography of Zone 11 (San Pedro Bay - Port of Los Angeles / Long Beach). The "Donut" strategy replaces the need to draw complex polygons by using the actual physical layout of the US Coast Guard's designated anchorage areas.

**The Physical Layout (The Proof):**

    2.1 The Inner Boundary ($< 3$ km) = Docked / Berthing:
    
    If we measure the distance from the main entrance of the LA Port (the breakwater) to the actual container cranes (like the APM Terminals or TraPac), the distance is roughly 2 to 3 kilometers. If a ship is under 3 km from the entrance and moving at $< 0.5$ knots, it is not "waiting"—it is actively tied to the dock unloading cargo.
    
    2.2 The Outer Boundary ($3$ km to $20$ km) = The Anchorage Zone:
    
    Because of the massive congestion during peak seasons, the Coast Guard established overflow anchorages. These anchorages start just outside the breakwater (about 3-4 km away) and extend all the way down the coast toward Huntington Beach and out toward Catalina Island. This designated waiting zone sits perfectly within that 3 km to 20 km radius.

Citations: 

NOAA Nautical Charts:

Look at NOAA Chart 18749 (San Pedro Bay). The designated anchorage areas (marked as Anchorage G, F, and the deep-water anchorages) form an arc outside the port entrance. If we overlay a scale, these anchorages fall exactly into the 2 to 12 nautical mile range (which is $\approx 3.7$ km to $22$ km).

Source: NOAA Office of Coast Survey, Chart 18749.

Marine Exchange of Southern California:The organization that manages the queue for LA/LB (The Marine Exchange) requires ships to wait in these specific outer anchorages when the port is congested. If a ship is outside 20-25 km, it is usually instructed to drift in the open ocean (often hundreds of miles away) to reduce coastal pollution, meaning it is not actively in the immediate "Port Queue."


In [ ]:
df_silver_clean = df_silver_clean.withColumn('is_in_waiting_area', \
                (F.col('sog') < 0.5) & 
                (F.col('distance_to_port') > 3) & 
                (F.col('distance_to_port') < 20) & 
                (F.col('status') == 1))

In [ ]:
df_silver_clean.groupBy("status").count().orderBy("count", ascending=False).show()

Status 1 = At Anchor (Waiting)

Status 5 = Moored (Tied to the dock)

Status 0 = Underway using engine (Moving)

3. **Bearing to Port (heading Error)**

- What is cog (Course Over Ground)? This is the direction the ship is currently traveling, measured in degrees from 0° to 359.9°, where 0° is True North, 90° is East, 180° is South, and 270° is West.

- What is "True Bearing"? True Bearing is the angle of the direct, straight line from the Ship to the Port, also measured relative to True North.

- The "Heading Error" (The Output): The Heading Error is simply the difference between the cog and the True Bearing.

    - Error = 0° : The ship is pointing perfectly at the port.

    - Error = 180° : The ship is sailing perfectly away from the port.

- If we want to fine the direction from point A to point B on a sphere, using Forward Azimuth formula. Related to Haversine formula, this formula is a little intense? Because Haversine formula is calculating the way going from the Port to the ship, and this Forward Azimuth is calculated the direction way from Ship to Port.

To calculate the Bearing ($\theta$), we first find two components, $y$ and $x$:

$$y = \sin(\Delta\lambda) \cdot \cos(\phi_2)$$

$$x = \cos(\phi_1) \cdot \sin(\phi_2) - \sin(\phi_1) \cdot \cos(\phi_2) \cdot \cos(\Delta\lambda)$$

$$\text{Bearing}(\theta) = \operatorname{atan2}(y, x)$$

$\phi_1, \lambda_1$: Ship's Latitude and Longitude.

$\phi_2, \lambda_2$: Port's Latitude and Longitude.

$\Delta\lambda$: The difference in Longitude ($\lambda_2 - \lambda_1$).

$y$ represents the "East-West" push towards the target.

$x$ represents the "North-South" push towards the target.

$\operatorname{atan2}$ is the "Brain." It takes those two pushes and figures out the exact angle on a $360^\circ$ circle. Unlike a regular atan(y/x), atan2 knows the difference between "North-East" and "South-West" because it looks at the plus/minus signs of $x$ and $y$ individually.


In [ ]:
df_silver_clean = df_silver_clean.withColumn('y_bearing', 
    F.sin(F.col('dlon')) * F.cos(F.lit(P_LAT))
).withColumn('x_bearing', 
    F.cos(F.col('lat_rad')) * F.sin(F.lit(P_LAT)) - 
    F.sin(F.col('lat_rad')) * F.cos(F.lit(P_LAT)) * F.cos(F.col('dlon'))
)

# 2. Calculate True Bearing in Radians, then convert to Degrees
df_silver_clean = df_silver_clean.withColumn('true_bearing_rad', 
    F.atan2(F.col('y_bearing'), F.col('x_bearing'))
).withColumn('true_bearing_deg', 
    # Convert to degrees and use modulo 360 to ensure it's always between 0 and 360
    (F.degrees(F.col('true_bearing_rad')) + 360) % 360
)

# 3. Calculate the Raw Error vs the ship's actual COG (Course Over Ground)
df_silver_clean = df_silver_clean.withColumn('raw_heading_error', 
    F.abs(F.col('cog') - F.col('true_bearing_deg'))
)

# 4. The "Circular Wrap" Fix (Ensure error is the shortest path, max 180 degrees)
df_silver_clean = df_silver_clean.withColumn('heading_error', 
    F.when(F.col('raw_heading_error') > 180, 360 - F.col('raw_heading_error'))
     .otherwise(F.col('raw_heading_error'))
)

# Optional: Drop the intermediate math columns to keep your dataframe clean
df_silver_clean = df_silver_clean.drop('y_bearing', 'x_bearing', 'true_bearing_rad', 'true_bearing_deg', 'raw_heading_error')

In [ ]:
df_silver_clean.columns

## 4. Systemic Congestion & Target Label Definition (`delay_minutes`)
This section defines the ultimate target variable. 
A vessel is considered "Waiting" if it is between 3km and 20km from the port moving at < 0.5 knots. The delay is the time difference between entering this waiting zone and finally anchoring/mooring at the berth.

### 4.1 Kinetic & Behavior Feature (Movement-Based)

Acceleration: $$a_i = \frac{SOG_i - SOG_{i-1}}{t_i - t_{i-1}}$$
Acceleration is calculated as the change in speed divided by the change in time
Source:\
[LJMU Research](https://researchonline.ljmu.ac.uk/id/eprint/21975/)\
[MDPI Maritime ML Papers](https://www.mdpi.com/2077-1312/14/8/712)


In [ ]:
# === KINETIC FEATURES (IMPROVED) ===
w = Window.partitionBy("mmsi").orderBy("base_date_time")
w3 = Window.partitionBy("mmsi").orderBy("base_date_time").rowsBetween(-2, 0)

df_silver_clean = df_silver_clean.withColumn("sog_smooth", 
    F.avg("sog").over(w3))
df_silver_clean = df_silver_clean.withColumn("prev_sog", 
    F.lag("sog_smooth", 1).over(w))
df_silver_clean = df_silver_clean.withColumn("time_diff_seconds",
    F.col("base_date_time").cast("long") 
    - F.lag("base_date_time", 1).over(w).cast("long"))
df_silver_clean = df_silver_clean.withColumn("acceleration",
    F.when(F.col("time_diff_seconds") >= 60,
        (F.col("sog_smooth") - F.col("prev_sog")) 
        / (F.col("time_diff_seconds") / 3600)
    ).otherwise(None))
df_silver_clean = df_silver_clean.drop("sog_smooth", "prev_sog")




"Dwell Time: the continuous duration a vessel spends within a defined port boundary or zone. AIS data allows researchers to calculate this by identifying the timestamp when a vessel enters a defined polygon and when it subsequently exits."\
[University of Arkansas Research](https://martrec.uark.edu/research/lsu_quantifying_final_report.pdf)


"Researchers often use speed-over-ground (SOG) thresholds (e.g., SOG ≤ 3.0 knots) to distinguish between vessels that are underway and those that are effectively stationary"\
[Frontiers in Marine Science](https://www.frontiersin.org/journals/future-transportation/articles/10.3389/ffutr.2025.1735788/full)

In [ ]:
df_silver_clean = df_silver_clean.withColumn("is_stationary", 
    F.when((F.col("sog") < 0.5), 1).otherwise(0))
df_silver_clean = df_silver_clean.withColumn("time_diff_hours",
    (F.col("base_date_time").cast("long") 
     - F.lag("base_date_time", 1).over(w).cast("long")) / 3600)
df_silver_clean = df_silver_clean.withColumn("stationary_hours",
    F.when(F.col("is_stationary") == 1, F.col("time_diff_hours")).otherwise(0))
df_silver_clean = df_silver_clean.withColumn("time_in_zone_hours",
    F.sum("stationary_hours").over(w.rowsBetween(Window.unboundedPreceding, 0)))

### 4.2. Systemic/Congestion Features (Context-Based)

Ship Density (Cargo/Tanker only)

"Spatial density measures how crowded the anchorage or terminal areas are at a given time." — [Kpler Congestion Analytics](https://www.kpler.com/blog/port-congestion-guide-measurement-causes-impact)

"Using the self-reported AIS Navigational Status to validate if a stationary vessel is actually performing a port operation." — [Frontiers in Marine Science](https://www.frontiersin.org/journals/future-transportation/articles/10.3389/ffutr.2025.1735788/full)

In [ ]:
df_silver_clean = df_silver_clean.withColumn(
    "hour_key", F.date_trunc("hour", F.col("base_date_time")))

df_density = (
df_silver_clean
.filter(
    (F.col("vessel_type").between(70, 79)) | 
    (F.col("vessel_type").between(80, 89))
)
.groupBy("hour_key")
.agg(F.countDistinct("mmsi").alias("ship_density"))
)

"Congestion Indices help monitor how long vessels spend at anchor versus how quickly they reach their berths, providing a measure of port congestion and infrastructure efficiency." — [OECD Port Statistics](https://www.oecd.org/en/data/dashboards/monitoring-maritime-trade-the-oecd-ais-vessel-tracking-dashboard.html)

In [ ]:
df_avg_speed = (
    df_silver_clean
    .filter(
        (F.col("vessel_type").between(70, 79)) | 
        (F.col("vessel_type").between(80, 89))
    )
    .groupBy("hour_key")
    .agg(F.avg("sog").alias("avg_port_speed"))
)


"Transition Points: Identifying the exact moments a vessel switches from 'underway' to 'at anchor' or 'moored' is critical for calculating port cycle times." 

"Berth Turnover Rate is calculated by tracking the frequency with which different vessels occupy a specific berth. AIS data allows analysts to identify when a berth becomes vacant and when a new vessel arrives." — [Frontiers in Marine Science](https://www.frontiersin.org/journals/future-transportation/articles/10.3389/ffutr.2025.1735788/full)

In [ ]:
w = Window.partitionBy("mmsi").orderBy("base_date_time")
df_with_prev_status = df_silver_clean.withColumn(
    "prev_status", F.lag("status", 1).over(w))
df_throughput = (
    df_with_prev_status
    .filter(
        (F.col("status") == 5) & 
        (F.col("prev_status") != 5)
    )
    .groupBy("hour_key")
    .agg(F.countDistinct("mmsi").alias("port_throughput"))
)

In [ ]:
df_silver_clean = (
    df_silver_clean
    .join(F.broadcast(df_density), on="hour_key", how="left")
    .join(F.broadcast(df_avg_speed), on="hour_key", how="left")
    .join(F.broadcast(df_throughput), on="hour_key", how="left")
)
df_silver_clean = df_silver_clean.fillna({"port_throughput": 0, 'ship_density': 0})

"A gap exceeding these thresholds is often assumed to represent a significant cessation of activity. Thresholds of 24 hours and 12 hours are frequently used as cut-off points to split trajectories." — [SciTePress Maritime Trajectory Analysis](https://www.scitepress.org/Papers/2022/109146/109146.pdf)

"Many algorithms split trajectories when the interval between consecutive AIS messages exceeds a specified threshold (e.g., 24 hours)." — [MDPI Maritime Traffic Patterns](https://www.mdpi.com/2673-7590/6/1/34)

In [ ]:
w = Window.partitionBy("mmsi").orderBy("base_date_time")

df_silver_clean = df_silver_clean.withColumn("prev_time", 
    F.lag("base_date_time", 1).over(w))
df_silver_clean = df_silver_clean.withColumn("gap_hours",
    (F.col("base_date_time").cast("long") 
     - F.col("prev_time").cast("long")) / 3600)

df_silver_clean = df_silver_clean.withColumn("is_new_visit",
    F.when(
        (F.col("gap_hours") > 24) | (F.col("prev_time").isNull()), 
        1
    ).otherwise(0))

"Waiting Time = (Timestamp of Arrival at Berth) – (Timestamp of Arrival at Anchorage)." — [Portcast](https://www.portcast.io/port-congestion/reykholar-7f4bc), [Kpler](https://www.kpler.com/blog/port-congestion-guide-measurement-causes-impact)

$$ \text{Delay}_{\text{minutes}} = \frac{(T_{\text{berth}} - T_{\text{entry}})_{\text{total\_seconds}}}{60} $$ 

— [Sinay Maritime Analytics](https://sinay.ai/en/eta-calculator-predicting-the-time-of-arrival-of-vessels/)

"$T_{\text{entry}}$: The timestamp of the first AIS message where the vessel enters the defined waiting zone. 

$T_{\text{berth}}$: The timestamp of the first AIS message where the vessel is detected within the defined berth polygon and exhibits 'moored' characteristics." 

— [Asian Development Bank Maritime Report](https://aric.adb.org/database/porttraffic/methodology)


"A gap exceeding these thresholds is often assumed to represent a significant cessation of activity. Thresholds of 24 hours and 12 hours are frequently used as cut-off points to split trajectories." — [SciTePress Maritime Trajectory Analysis](https://www.scitepress.org/Papers/2022/109146/109146.pdf)

"Many algorithms split trajectories when the interval between consecutive AIS messages exceeds a specified threshold (e.g., 24 hours)." — [MDPI Maritime Traffic Patterns](https://www.mdpi.com/2673-7590/6/1/34)

In [ ]:
# === VISIT SEGMENTATION + DELAY (ROBUST MOORING) ===
df_silver_clean = df_silver_clean.withColumn("visit_id",
    F.sum("is_new_visit").over(w.rowsBetween(Window.unboundedPreceding, 0)))

df_entry = (df_silver_clean
    .groupBy("mmsi", "visit_id")
    .agg(F.min("base_date_time").alias("t_enter")))

df_moored = (df_silver_clean
    .filter(
        (F.col("status") == 5) |
        ((F.col("sog") < 0.5) & (F.col("distance_to_port") < 2))
    )
    .groupBy("mmsi", "visit_id")
    .agg(F.min("base_date_time").alias("t_moored")))

df_delay = (df_entry
    .join(df_moored, on=["mmsi", "visit_id"], how="left")
    .withColumn("delay_minutes",
        F.when(F.col("t_moored").isNotNull(),
            (F.col("t_moored").cast("long") 
             - F.col("t_enter").cast("long")) / 60
        ).otherwise(None))
    .select("mmsi", "visit_id", "delay_minutes"))

df_silver_clean = df_silver_clean.join(
    F.broadcast(df_delay), on=["mmsi", "visit_id"], how="left")

# avg_waiting_time_last_24h
df_moored_events = (df_silver_clean
    .filter(
        (F.col("status") == 5) |
        ((F.col("sog") < 0.5) & (F.col("distance_to_port") < 2))
    )
    .groupBy("mmsi", "visit_id")
    .agg(
        F.min("base_date_time").alias("moored_time"),
        F.first("delay_minutes", ignorenulls=True).alias("vessel_delay")
    )
    .filter(F.col("vessel_delay").isNotNull())
    .withColumn("moored_hour", F.date_trunc("hour", F.col("moored_time")))
)

df_delay_per_hour = (df_moored_events
    .groupBy("moored_hour")
    .agg(
        F.avg("vessel_delay").alias("delay_that_hour"),
        F.count("*").alias("ships_moored_that_hour")
    ))

w_hour = (Window
    .orderBy(F.col("moored_hour").cast("long"))
    .rangeBetween(-24 * 3600, -1))

df_avg_wait = (df_delay_per_hour
    .withColumn("avg_waiting_time_last_24h", 
        F.avg("delay_that_hour").over(w_hour))
    .select("moored_hour", "avg_waiting_time_last_24h")
    .withColumnRenamed("moored_hour", "hour_key"))

df_silver_clean = df_silver_clean.join(
    F.broadcast(df_avg_wait), on="hour_key", how="left")


"Statistical Metrics: Beyond simple averages, platforms calculate percentiles (e.g., P90 waiting time) to highlight 'long-tail' congestion." — [Portcast](https://www.portcast.io/port-congestion/reykholar-7f4bc)

"Average/Median waiting time used as a baseline for comparing current port performance against historical norms." — [Portcast Port Analytics](https://www.portcast.io/blog/port-congestion-snapshot)

In [ ]:
df_silver_clean.columns

In [ ]:
# total = df_silver_clean.count()
# print(f"Total rows: {total}\n")

# for col_name in df_silver_clean.columns:
#     null_count = df_silver_clean.filter(F.col(col_name).isNull()).count()
#     if null_count > 0:
#         pct = round(null_count / total * 100, 2)
#         print(f"  {col_name}: {null_count} nulls ({pct}%)")


In [ ]:
df_silver_clean = df_silver_clean.fillna({"time_in_zone_hours": 0})

In [ ]:
intermediate_cols = [
    'lat_rad', 'lon_rad', 'dlat', 'dlon', 'a', 'c',
    'prev_sog', 'time_diff_seconds',
    'is_stationary', 'time_diff_hours', 'stationary_hours',
    'prev_time', 'gap_hours', 'is_new_visit', 'avg_waiting_time_last_24h'
]

df_export = df_silver_clean.drop(*intermediate_cols)
print(f"Final: {len(df_export.columns)} columns")
# Expected: 48 columns


In [ ]:
df_visit_check = (df_silver_clean
    .groupBy("mmsi", "visit_id")
    .agg(F.first("delay_minutes").alias("delay"))
)

total_visits = df_visit_check.count()
moored_visits = df_visit_check.filter(F.col("delay").isNotNull()).count()

print(f"Total visits: {total_visits}")
print(f"Moored visits: {moored_visits} ({round(moored_visits/total_visits*100,1)}%)")
print(f"Never moored: {total_visits - moored_visits} ({round((total_visits-moored_visits)/total_visits*100,1)}%)")


# Crawling Data for automate

In [ ]:
output_path = f"Files/Gold/ais_zone11_{batch_name}_preprocessed.parquet"
df_export.coalesce(1).write.mode("overwrite").parquet(output_path)
print(f"Exported: {output_path}")
